# Network Reconstruction Evaluation

Comprehensive analysis of the reconstructed supplier-user network including degree distributions, NACE category connections, and spatial distances.

## Section 1: Load Data and Build Directed Network

Load node and edge datasets, validate required columns, and construct a directed graph structure for analysis.

In [1]:
# Import Required Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import networkx as nx
from collections import defaultdict

# Set style for better plots
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

Matplotlib is building the font cache; this may take a moment.


In [ ]:
# Create histograms for distance distributions
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Turnover distance histogram
axes[0].hist(reconstructed_edges['to_distance'], bins=50, color='steelblue', edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Absolute Turnover Difference', fontsize=11)
axes[0].set_ylabel('Frequency', fontsize=11)
axes[0].set_title('Distribution of Turnover Distance\nBetween Suppliers and Users', fontsize=12, fontweight='bold')
axes[0].grid(True, alpha=0.3)
to_dist = reconstructed_edges['to_distance']
axes[0].axvline(to_dist.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {to_dist.mean():.2e}')
axes[0].axvline(to_dist.median(), color='green', linestyle='--', linewidth=2, label=f'Median: {to_dist.median():.2e}')
axes[0].legend()

# Wages distance histogram
axes[1].hist(reconstructed_edges['wages_distance'], bins=50, color='coral', edgecolor='black', alpha=0.7)
axes[1].set_xlabel('Absolute Wages Difference', fontsize=11)
axes[1].set_ylabel('Frequency', fontsize=11)
axes[1].set_title('Distribution of Wages Distance\nBetween Suppliers and Users', fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3)
wages_dist = reconstructed_edges['wages_distance']
axes[1].axvline(wages_dist.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {wages_dist.mean():.2e}')
axes[1].axvline(wages_dist.median(), color='green', linestyle='--', linewidth=2, label=f'Median: {wages_dist.median():.2e}')
axes[1].legend()

# Composite distance histogram (normalized)
axes[2].hist(reconstructed_edges['composite_distance'], bins=50, color='lightgreen', edgecolor='black', alpha=0.7)
axes[2].set_xlabel('Composite Distance (normalized)', fontsize=11)
axes[2].set_ylabel('Frequency', fontsize=11)
axes[2].set_title('Distribution of Composite Distance\n(Turnover + Wages)', fontsize=12, fontweight='bold')
axes[2].grid(True, alpha=0.3)
comp_dist = reconstructed_edges['composite_distance']
axes[2].axvline(comp_dist.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {comp_dist.mean():.3f}')
axes[2].axvline(comp_dist.median(), color='green', linestyle='--', linewidth=2, label=f'Median: {comp_dist.median():.3f}')
axes[2].legend()

plt.tight_layout()
plt.show()

print("Distance distribution visualizations complete.")

## Summary Statistics

Network reconstruction evaluation complete. Key metrics computed:
- In-degree and out-degree distributions
- NACE category connection matrix
- Supplier-user economic distance distributions

In [ ]:
# Calculate various distance metrics between suppliers and users
reconstructed_edges['to_distance'] = abs(reconstructed_edges['user_to'] - reconstructed_edges['supplier_to'])
reconstructed_edges['wages_distance'] = abs(reconstructed_edges['user_wages'] - reconstructed_edges['supplier_wages'])

# Normalize distances to 0-1 scale for comparison
to_distance_normalized = (reconstructed_edges['to_distance'] - reconstructed_edges['to_distance'].min()) / \
                         (reconstructed_edges['to_distance'].max() - reconstructed_edges['to_distance'].min() + 1e-10)
wages_distance_normalized = (reconstructed_edges['wages_distance'] - reconstructed_edges['wages_distance'].min()) / \
                            (reconstructed_edges['wages_distance'].max() - reconstructed_edges['wages_distance'].min() + 1e-10)

reconstructed_edges['composite_distance'] = (to_distance_normalized + wages_distance_normalized) / 2

print("Distance metrics computed:")
print(f"\nTurnover (TO) Distance Statistics:")
print(f"  Min: {reconstructed_edges['to_distance'].min():.2e}")
print(f"  Mean: {reconstructed_edges['to_distance'].mean():.2e}")
print(f"  Median: {reconstructed_edges['to_distance'].median():.2e}")
print(f"  Max: {reconstructed_edges['to_distance'].max():.2e}")

print(f"\nWages Distance Statistics:")
print(f"  Min: {reconstructed_edges['wages_distance'].min():.2e}")
print(f"  Mean: {reconstructed_edges['wages_distance'].mean():.2e}")
print(f"  Median: {reconstructed_edges['wages_distance'].median():.2e}")
print(f"  Max: {reconstructed_edges['wages_distance'].max():.2e}")

print(f"\nComposite Distance Statistics (normalized):")
print(f"  Min: {reconstructed_edges['composite_distance'].min():.3f}")
print(f"  Mean: {reconstructed_edges['composite_distance'].mean():.3f}")
print(f"  Median: {reconstructed_edges['composite_distance'].median():.3f}")
print(f"  Max: {reconstructed_edges['composite_distance'].max():.3f}")

In [ ]:
# Display NACE links as sorted table
print("=" * 80)
print("NACE-to-NACE Connections Table (sorted by link count)")
print("=" * 80)
display_nace = nace_links.copy()
display_nace.columns = ['Supplier NACE', 'User NACE', 'Link Count']
print(display_nace.to_string(index=False))

# Calculate summary statistics
total_links_by_supplier = nace_links.groupby('supplier_nace')['link_count'].sum().sort_values(ascending=False)
total_links_by_user = nace_links.groupby('user_nace')['link_count'].sum().sort_values(ascending=False)

print("\n\n" + "=" * 80)
print("Total Links by Supplier NACE (top 10)")
print("=" * 80)
print(total_links_by_supplier.head(10))

print("\n\n" + "=" * 80)
print("Total Links by User NACE (top 10)")
print("=" * 80)
print(total_links_by_user.head(10))

In [ ]:
# Load enterprise data (nodes) and reconstructed network (edges)
try:
    # Try with duckdb first, then pandas
    import duckdb
    
    enterprises = duckdb.sql("SELECT * FROM read_parquet('data-raw/data.parquet')").df()
    reconstructed_edges = pd.read_parquet('data/reconstructed_network.parquet')
except Exception as e:
    print(f"Error loading with duckdb: {e}")
    # Fallback to direct pandas
    enterprises = pd.read_parquet('data-raw/data.parquet')
    reconstructed_edges = pd.read_parquet('data/reconstructed_network.parquet')

print(f"Enterprises shape: {enterprises.shape}")
print(f"Reconstructed edges shape: {reconstructed_edges.shape}")
print(f"\nEnterprises columns: {list(enterprises.columns)}")
print(f"Edges columns: {list(reconstructed_edges.columns)}")
print(f"\nFirst few edges:\n{reconstructed_edges.head()}")

## Section 2: Compute In-Degree and Out-Degree

Calculate in-degree and out-degree for each node and store results in a dataframe.

In [ ]:
# Create a mapping from enterprise ID to NACE category
id_to_nace = dict(zip(enterprises['id'], enterprises['NACE']))
id_to_to = dict(zip(enterprises['id'], enterprises['TO']))
id_to_wages = dict(zip(enterprises['id'], enterprises['WAGES']))

# Add NACE information to edges
reconstructed_edges['user_nace'] = reconstructed_edges['user_id'].map(id_to_nace)
reconstructed_edges['supplier_nace'] = reconstructed_edges['supplier_id'].map(id_to_nace)
reconstructed_edges['user_to'] = reconstructed_edges['user_id'].map(id_to_to)
reconstructed_edges['supplier_to'] = reconstructed_edges['supplier_id'].map(id_to_to)
reconstructed_edges['user_wages'] = reconstructed_edges['user_id'].map(id_to_wages)
reconstructed_edges['supplier_wages'] = reconstructed_edges['supplier_id'].map(id_to_wages)

# Calculate distances (using turnover difference as a proxy for economic distance)
reconstructed_edges['to_distance'] = abs(reconstructed_edges['user_to'] - reconstructed_edges['supplier_to'])
reconstructed_edges['wages_distance'] = abs(reconstructed_edges['user_wages'] - reconstructed_edges['supplier_wages'])

# Create directed network
G = nx.DiGraph()
G.add_edges_from(zip(reconstructed_edges['user_id'], reconstructed_edges['supplier_id']))

print(f"Network created with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges")

## Section 3: Plot In-Degree and Out-Degree Histograms

Visualize the distribution of in-degree and out-degree across all nodes.

In [ ]:
# Calculate in-degree and out-degree for each node
in_degree = dict(G.in_degree())
out_degree = dict(G.out_degree())

# Create degree dataframe
degree_df = pd.DataFrame({
    'node_id': list(G.nodes()),
    'in_degree': [in_degree.get(n, 0) for n in G.nodes()],
    'out_degree': [out_degree.get(n, 0) for n in G.nodes()]
})

# Add NACE category information
degree_df['nace'] = degree_df['node_id'].map(id_to_nace)

print(f"Degree statistics:")
print(f"\nIn-degree:")
print(degree_df['in_degree'].describe())
print(f"\nOut-degree:")
print(degree_df['out_degree'].describe())
print(f"\nDegree dataframe head:\n{degree_df.head(10)}")

## Section 4: Aggregate Links Between NACE Categories

Count connections between source and target NACE categories.

In [ ]:
# Create side-by-side histograms for in-degree and out-degree
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# In-degree histogram
axes[0].hist(degree_df['in_degree'], bins=30, color='steelblue', edgecolor='black', alpha=0.7)
axes[0].set_xlabel('In-Degree (number of suppliers)', fontsize=11)
axes[0].set_ylabel('Frequency', fontsize=11)
axes[0].set_title('Distribution of In-Degree\n(How many suppliers per user)', fontsize=12, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Add statistics annotation
in_stats = degree_df['in_degree']
axes[0].axvline(in_stats.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {in_stats.mean():.2f}')
axes[0].axvline(in_stats.median(), color='green', linestyle='--', linewidth=2, label=f'Median: {in_stats.median():.2f}')
axes[0].legend()

# Out-degree histogram
axes[1].hist(degree_df['out_degree'], bins=30, color='coral', edgecolor='black', alpha=0.7)
axes[1].set_xlabel('Out-Degree (number of customers)', fontsize=11)
axes[1].set_ylabel('Frequency', fontsize=11)
axes[1].set_title('Distribution of Out-Degree\n(How many users per supplier)', fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3)

# Add statistics annotation
out_stats = degree_df['out_degree']
axes[1].axvline(out_stats.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {out_stats.mean():.2f}')
axes[1].axvline(out_stats.median(), color='green', linestyle='--', linewidth=2, label=f'Median: {out_stats.median():.2f}')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"\nIn-Degree Summary:")
print(f"  Mean: {in_stats.mean():.3f}")
print(f"  Median: {in_stats.median():.3f}")
print(f"  Std Dev: {in_stats.std():.3f}")
print(f"  Min: {in_stats.min():.0f}")
print(f"  Max: {in_stats.max():.0f}")

print(f"\nOut-Degree Summary:")
print(f"  Mean: {out_stats.mean():.3f}")
print(f"  Median: {out_stats.median():.3f}")
print(f"  Std Dev: {out_stats.std():.3f}")
print(f"  Min: {out_stats.min():.0f}")
print(f"  Max: {out_stats.max():.0f}")

## Section 5: Display NACE-to-NACE Connections Table

Present the aggregated NACE link counts as both a sorted table and a heatmap.

In [ ]:
# Aggregate links between NACE categories
nace_links = reconstructed_edges.groupby(['supplier_nace', 'user_nace']).size().reset_index(name='link_count')
nace_links = nace_links.sort_values('link_count', ascending=False)

print(f"Total NACE pairs with links: {len(nace_links)}")
print(f"\nTop 15 NACE-to-NACE connections:")
print(nace_links.head(15).to_string(index=False))

# Create pivot table for NACE matrix
nace_matrix = nace_links.pivot_table(
    index='supplier_nace',
    columns='user_nace',
    values='link_count',
    fill_value=0,
    aggfunc='sum'
)

print(f"\nNACE matrix shape: {nace_matrix.shape}")

## Section 6: Compute Supplier-to-User Distances

Calculate distance between suppliers and users using multiple metrics.

In [ ]:
# Create heatmap of NACE connections
plt.figure(figsize=(16, 12))
sns.heatmap(nace_matrix, cmap='YlOrRd', annot=True, fmt='.0f', cbar_kws={'label': 'Number of Links'})
plt.title('NACE-to-NACE Connection Matrix\n(Rows: Suppliers, Columns: Users)', fontsize=14, fontweight='bold')
plt.xlabel('User NACE Category', fontsize=11)
plt.ylabel('Supplier NACE Category', fontsize=11)
plt.tight_layout()
plt.show()

## Section 7: Plot Supplier-to-User Distance Histogram

Visualize the distribution of distances between suppliers and users.